# Diagnose collection size, representation and additional information
**Objective:** distinguish three controlled questions and interpret their uncertainty. These analyses are exploratory because the earlier tests were already known.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, SVG
ROOT = Path.cwd() if (Path.cwd() / 'data/manifest.json').exists() else Path.cwd().parent
assert (ROOT / 'data/manifest.json').exists(), 'Run from the project root or notebooks directory.'
sys.path.insert(0, str(ROOT / 'scripts'))
PILOT = ROOT / 'results/pilot_v1'
MATCHED = ROOT / 'results/transition_v2'
GUIDE = ROOT / 'results/guide_v3'


## 1. Adding cells is different from adding correlated rows

In [ ]:
m=pd.read_csv(GUIDE/'metrics.csv')
subsets=pd.read_csv(GUIDE/'learning_subsets.csv.gz',dtype={'row_id':str})
for seed in [17,42,73]:
    s=[set(subsets.loc[(subsets.seed==seed)&(subsets.fraction==f),'row_id']) for f in [.25,.5,1.]]
    assert s[0]<s[1]<s[2]
display(m[m.case.str.startswith('learning_')].groupby(['fraction','model'])[['n_train','training_cells','mae']].mean())


## 2. Inspect exactly what the missing-value representation changes

In [ ]:
base=np.load(MATCHED/'antwerp_spatial_500m/examples.npz',allow_pickle=False)
representation=np.load(GUIDE/'missing_representation.npz',allow_pickle=False)
print('Original shape:',base['X'].shape,'Mean + mask:',representation['X'].shape)
print('One representation row:',representation['X'][0])
display(m[m.case.str.startswith('missing_')].groupby(['case','model']).mae.agg(['mean','std']))


## 3. Pair comparisons on the same held-out observations
The intervals resample days, cells or addresses. They condition on these fitted models and available blocks; they are not confidence about a new city.

In [ ]:
paired=pd.read_csv(GUIDE/'paired_differences.csv')
display(paired)
environment=m[m.case.str.startswith('environment_')]
display(environment.groupby(['case','model']).rmse.agg(['mean','std']))


**Exercise:** which source of variability changes in a learning curve that varies regions and neural seeds together? Does an environmental ablation identify a causal effect?

<details><summary>Feedback after your attempt</summary>

both sampled training support and optimization vary. An observational ablation measures incremental predictive information, with possible proxies and confounding. A causal claim requires a design that identifies the intervention of interest.

</details>

**Adaptation task:** freeze one new experiment that varies only information, only model structure or only collection units; state which factors cannot be held equal.